# Xenium lymph node processing

This notebook turns the SpatialData object written by the sopa workflow into an
AnnData table ready for clustering and spatial analysis.

It expects the proseg output (`config/sopa_proseg.yaml`) with raw transcript counts
in `table.X`, and reads every path and parameter from `config/xenium.yaml`.

Steps: read the zarr, keep raw counts, log-normalise, filter low-count cells,
select highly variable genes, PCA, neighbour graph, UMAP and Leiden at several
resolutions, then write the h5ad.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import scanpy as sc
import spatialdata as sd
import yaml

# Accept both "run from the repository root" and "run from notebooks/".
root = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "config" / "xenium.yaml").exists()
)
CONFIG_PATH = root / "config" / "xenium.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg

In [ ]:
sdata = sd.read_zarr(root / cfg["input"]["proseg_zarr"])
sdata

In [ ]:
adata = sdata.tables["table"]

# X holds raw transcript counts. Keep them, then normalise in place.
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

adata

In [ ]:
sc.pp.filter_cells(adata, min_counts=cfg["qc"]["min_counts"])

max_cells = cfg["qc"].get("max_cells")
if max_cells is not None and adata.n_obs > max_cells:
    sc.pp.subsample(adata, n_obs=int(max_cells), random_state=0)

print(f"{adata.n_obs} cells, {adata.n_vars} genes")

In [ ]:
clu = cfg["clustering"]
sc.pp.highly_variable_genes(adata, n_top_genes=clu["n_top_genes"], flavor="seurat")
sc.pp.pca(adata, n_comps=clu["n_pcs"], mask_var="highly_variable")
sc.pp.neighbors(adata, n_neighbors=clu["n_neighbors"], n_pcs=clu["n_pcs"])
sc.tl.umap(adata)

for res in clu["resolutions"]:
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=f"{clu['key_prefix']}_{res}",
        flavor="igraph",
        n_iterations=2,
        directed=False,
    )

adata

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color=f"{clu['key_prefix']}_{clu['resolutions'][1]}",
                frameon=False, legend_loc="on data", legend_fontsize=6, title="UMAP")

In [ ]:
sc.pl.embedding(adata, basis="spatial", color=f"{clu['key_prefix']}_{clu['resolutions'][1]}",
                frameon=False, size=2, title="Spatial clustering")

In [ ]:
out = root / cfg["output"]["h5ad"]
out.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(out)
print(f"written {out}")